# Notebook 4 — Style Transfer & Portfolio Generation

**Phase 3/4** deliverable. This notebook focuses on applying your
CycleGAN/Pix2Pix models to real style-transfer data (e.g. WikiArt), then
using the platform's export pipeline to build your 50+ image Generated Art
Portfolio (`outputs/portfolio/`).

Make sure `pytest -m advanced -v` and `pytest -m platform -v` pass before
relying on the helpers used here.


In [1]:
import sys
from pathlib import Path

# Make `generative_art_studio` importable without `pip install -e .`
REPO_ROOT = Path.cwd().parent if (Path.cwd() / "notebooks").exists() else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

import torch
# Ensure the project root is used when the notebook is launched from the notebooks folder.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = REPO_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from generative_art_studio.utils import set_seed, plot_image_grid
from generative_art_studio.config import DEVICE

set_seed(42)
print(f"Using device: {DEVICE}")


Using device: cpu


## 1. Load a style-transfer dataset

Download WikiArt (or another style dataset) first — see `docs/SETUP.md`
and `scripts/download_data.py --dataset wikiart --out data/wikiart`.


In [2]:
from generative_art_studio.data import get_image_dataset, get_dataloader

# TODO: point this at your downloaded style dataset once available.
# style_dataset = get_image_dataset("data/wikiart", image_size=64)
# style_dataloader = get_dataloader(style_dataset, batch_size=16)
#download the dataset from https://www.kaggle.com/datasets/ikarus777/best-artworks-of-all-time
download_path = "data/wikiart"


style_dataset = get_image_dataset("data/wikiart", image_size=64)
style_dataloader = get_dataloader(style_dataset, batch_size=16)


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'data/wikiart'

## 2. Apply your trained CycleGAN/Pix2Pix generator

Load a checkpoint you saved while training in Notebook 3, run it over a
batch of content images, and visualize input vs. stylized output
side-by-side.


In [3]:
# TODO: load a trained generator checkpoint and run style transfer here
# e.g.
# from generative_art_studio.models.advanced import CycleGANGenerator
# generator = CycleGANGenerator(features=32, num_residual_blocks=4).to(DEVICE)
# generator.load_state_dict(torch.load("checkpoints/cyclegan_a2b.pt", map_location=DEVICE))
# generator.eval()
from generative_art_studio.models.advanced import CycleGANGenerator

generator = CycleGANGenerator(features=32, num_residual_blocks=4).to(DEVICE)
generator.load_state_dict(torch.load("checkpoints/cyclegan_a2b.pt", map_location=DEVICE))
generator.eval()


FileNotFoundError: [Errno 2] No such file or directory: 'checkpoints/cyclegan_a2b.pt'

## 3. Style mixing & interpolation

Reuse `interpolate_latent` (VAE) or interpolate between two *content*
images fed through your style-transfer generator to create a smooth
transition sequence — a nice portfolio piece.


In [4]:
# TODO: build an interpolation grid, e.g. blend two source images before
# feeding them through your generator, or interpolate a VAE latent path
# and decode each step (see Notebook 1).
interpolation_steps = 10
interpolation_grid = []

# Ensure style_dataloader is available
if 'style_dataloader' not in locals():
    style_dataloader = get_dataloader(style_dataset, batch_size=16)

image1, _ = next(iter(style_dataloader))
image2, _ = next(iter(style_dataloader))
generated_images = []

for i in range(interpolation_steps):
    alpha = i / (interpolation_steps - 1)
    blended_image = (1 - alpha) * image1 + alpha * image2
    blended_image = blended_image.to(DEVICE)
    with torch.no_grad():
        generated_image = generator(blended_image)
    generated_images.append(generated_image.cpu())




NameError: name 'style_dataset' is not defined

## 4. Build your Generated Art Portfolio

Use `generative_art_studio.app.model_registry` — the same code your
Streamlit platform (Phase 4) uses — to generate and export high-resolution
images into `outputs/portfolio/`. Implement `generate_samples` and
`export_image` in `app/model_registry.py` first (`pytest -m platform -v`).


In [5]:
from generative_art_studio.app.model_registry import load_model, generate_samples, export_image

vae_model = load_model("vae", device=DEVICE)
gan_model = load_model("vanilla_gan", device=DEVICE)

portfolio_dir = REPO_ROOT / "outputs" / "portfolio"
count = 0
for model_key, model in [("vae", vae_model), ("vanilla_gan", gan_model)]:
    images = generate_samples(model_key, model, num_samples=25, seed=count, device=DEVICE)
    for i, img in enumerate(images):
        export_image(img, portfolio_dir / f"{model_key}_samples" / f"{model_key}_{i:03d}.png", scale_factor=4)
        count += 1

print(f"Exported {count} portfolio images to {portfolio_dir}")


Exported 50 portfolio images to c:\Users\jbisw\OneDrive - UNT System\Documents\Project5\generative-art-studio-tanvibbsr\outputs\portfolio


## Reflection (for your Technical Report / Presentation)

- Which model produced your favorite portfolio pieces, and why (artistic
  judgment, not just metrics)?
- What would you improve about the style-transfer results with more
  training time or a different architecture?
- Record your 10–12 minute video presentation demonstrating the platform
  (`streamlit run src/generative_art_studio/app/streamlit_app.py`) and
  walking through this portfolio.
